In [2]:
import requests
import pandas as pd
import os

# save folder
SAVE_DIR = r"E:\Pred_Market"

# create folder if missing
os.makedirs(SAVE_DIR, exist_ok=True)

# API endpoint
url = "https://gamma-api.polymarket.com/markets"

# request data
r = requests.get(url)

# convert json
data = r.json()

rows = []

# loop through markets
for m in data:

    row = {
        "market_id": m.get("id"),
        "question": m.get("question"),
        "active": m.get("active"),
        "closed": m.get("closed"),
        "end_date": m.get("endDate"),
        "volume": m.get("volume"),
        "liquidity": m.get("liquidity"),
    }

    rows.append(row)

# dataframe
df = pd.DataFrame(rows)

# output file
save_path = os.path.join(SAVE_DIR, "polymarket_markets.csv")

# save csv
df.to_csv(save_path, index=False)

print(df.head())

print("\nSaved to:")
print(save_path)

print("\nTotal markets:", len(df))

  market_id                                  question  active  closed  \
0    540817          New Rihanna Album before GTA VI?    True   False   
1    540818    New Playboi Carti Album before GTA VI?    True   False   
2    540819   Will Jesus Christ return before GTA VI?    True   False   
3    540820     Trump out as President before GTA VI?    True   False   
4    540843  Will China invades Taiwan before GTA VI?    True   False   

               end_date              volume    liquidity  
0  2026-07-31T12:00:00Z   766438.8140969981   23173.1434  
1  2026-07-31T12:00:00Z   736752.9116369466   22047.8506  
2  2026-07-31T12:00:00Z  11482582.294408772  449057.2456  
3  2026-07-31T12:00:00Z   658436.4439350074   32976.0837  
4  2026-07-31T12:00:00Z  1843865.9200210618    51498.863  

Saved to:
E:\Pred_Market\polymarket_markets.csv

Total markets: 20


In [ ]:
import requests
import pandas as pd
import os
import time
import ast

from tqdm import tqdm
from datetime import datetime

# =====================================
# SETTINGS
# =====================================

BASE_DIR = r"E:\Pred_Market"

MARKETS_DIR = os.path.join(BASE_DIR, "markets")

os.makedirs(MARKETS_DIR, exist_ok=True)

# global index
index_file = os.path.join(
    BASE_DIR,
    "market_index.csv"
)

# API
url = "https://gamma-api.polymarket.com/markets"

# collect every 5 mins
SLEEP_TIME = 300

# total runtime (2 hours)
MAX_RUNTIME = 2 * 60 * 60

# =====================================
# LOAD INDEX
# =====================================

if os.path.exists(index_file):

    index_df = pd.read_csv(index_file)

else:

    index_df = pd.DataFrame(
        columns=[
            "num_id",
            "market_id",
            "question"
        ]
    )

# =====================================
# GET OR CREATE MARKET ID
# =====================================

def get_num_id(market_id, question):

    global index_df

    m = index_df[
        index_df["market_id"] == str(market_id)
    ]

    # existing
    if len(m) > 0:

        return int(m.iloc[0]["num_id"])

    # new market
    new_id = len(index_df)

    row = {
        "num_id": new_id,
        "market_id": str(market_id),
        "question": question
    }

    index_df.loc[len(index_df)] = row

    # save
    index_df.to_csv(index_file, index=False)

    return new_id

# =====================================
# START TIMER
# =====================================

start_time = time.time()

# =====================================
# MAIN LOOP
# =====================================

while True:

    # =====================================
    # CHECK TOTAL RUNTIME
    # =====================================

    elapsed = time.time() - start_time

    if elapsed > MAX_RUNTIME:

        print("\nReached 2-hour runtime.")
        print("Stopping safely.\n")

        break

    print("\nCollecting market data...")

    ts = datetime.utcnow().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    # =====================================
    # REQUEST WITH RETRY
    # =====================================

    try:

        r = requests.get(
            url,
            timeout=30
        )

        data = r.json()

    except Exception as e:

        print("\nRequest failed:", e)

        print("Sleeping 1 minute...\n")

        time.sleep(60)

        continue

    # =====================================
    # LOOP MARKETS
    # =====================================

    for m in tqdm(data):

        try:

            market_id = str(m.get("id"))

            question = m.get("question")

            # get numeric id
            num_id = get_num_id(
                market_id,
                question
            )

            # =====================================
            # MARKET FOLDER
            # =====================================

            market_folder = os.path.join(
                MARKETS_DIR,
                str(num_id)
            )

            os.makedirs(
                market_folder,
                exist_ok=True
            )

            # =====================================
            # INFO FILE
            # =====================================

            info_file = os.path.join(
                market_folder,
                "info.csv"
            )

            # save once
            if not os.path.exists(info_file):

                info_row = {
                    "market_id": market_id,
                    "question": question,
                    "end_date": m.get("endDate"),
                    "active": m.get("active"),
                    "closed": m.get("closed")
                }

                pd.DataFrame(
                    [info_row]
                ).to_csv(
                    info_file,
                    index=False
                )

            # =====================================
            # PARSE PRICES
            # =====================================

            yes_price = None
            no_price = None

            prices = m.get("outcomePrices")

            if prices is not None:

                if isinstance(prices, str):

                    prices = ast.literal_eval(prices)

                if len(prices) >= 1:
                    yes_price = float(prices[0])

                if len(prices) >= 2:
                    no_price = float(prices[1])

            # =====================================
            # HISTORY FILE
            # =====================================

            history_file = os.path.join(
                market_folder,
                "history.csv"
            )

            row = {
                "timestamp": ts,
                "yes_price": yes_price,
                "no_price": no_price,
                "volume": m.get("volume"),
                "liquidity": m.get("liquidity")
            }

            df = pd.DataFrame([row])

            # append
            if os.path.exists(history_file):

                df.to_csv(
                    history_file,
                    mode="a",
                    header=False,
                    index=False
                )

            else:

                df.to_csv(
                    history_file,
                    index=False
                )

        except Exception as e:

            print("Market error:", e)

    print("\nDone.")

    # =====================================
    # WAIT
    # =====================================

    mins = SLEEP_TIME // 60

    print(f"\nSleeping {mins} minutes...\n")

    time.sleep(SLEEP_TIME)

# =====================================
# FINISHED
# =====================================

print("Session complete.")

C:\Users\Shaif\AppData\Local\Temp\ipykernel_7796\1405526048.py:113: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime(
100%|█████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 161.05it/s]



Done.

Sleeping 5 minutes...




C:\Users\Shaif\AppData\Local\Temp\ipykernel_7796\1405526048.py:113: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  ts = datetime.utcnow().strftime(
100%|█████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 371.98it/s]



Done.

Sleeping 5 minutes...




100%|█████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 537.23it/s]


...


Done.

Sleeping 5 minutes...




100%|█████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 432.03it/s]



Done.

Sleeping 5 minutes...




100%|█████████████████████████████████████████████████████████████████████████████████| 20/20 [00:00<00:00, 442.04it/s]



Done.

Sleeping 5 minutes...

